### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Tesla Laptops\AppData\Local\Temp\ipykernel_20444\3424703257.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [6]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")

Found 2 PDF files to process

Processing: hackathonguide.pdf
  ✓ Loaded 8 pages

Processing: WellAbroad — System Architecture & Implementation Guide.pdf
  ✓ Loaded 13 pages

Total documents loaded: 21


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-10-19T00:07:48+05:00', 'author': 'fizzah', 'moddate': '2026-10-19T00:07:48+05:00', 'title': 'Microsoft Word - HackInsight_Participant_Guide_7pages', 'source': '..\\data\\pdf\\hackathonguide.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'hackathonguide.pdf', 'file_type': 'pdf'}, page_content='How to use this PDF \nBlue underlined text is a clickable link, both inside the guide and to websites. Click any entry in the Contents, or \'Go to Contents\' at the top of a page, to jump. \nOpen the bookmarks panel of your PDF viewer to move between chapters. Tick the boxes and type in the blue fill-in fields. Use Adobe Acrobat, Chrome or \nEdge so that your answers are kept when you save the file. \nEach chapter also appears in the bookmarks panel. Type your answers in the blue boxes; they are saved in the file when you use Adobe Acrobat, Chrome or \nEdge. \n \nFrom a real p

In [8]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [9]:
chunks=split_documents(all_pdf_documents)
chunks

Split 21 documents into 72 chunks

Example chunk:
Content: How to use this PDF 
Blue underlined text is a clickable link, both inside the guide and to websites. Click any entry in the Contents, or 'Go to Contents' at the top of a page, to jump. 
Open the book...
Metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-10-19T00:07:48+05:00', 'author': 'fizzah', 'moddate': '2026-10-19T00:07:48+05:00', 'title': 'Microsoft Word - HackInsight_Participant_Guide_7pages', 'source': '..\\data\\pdf\\hackathonguide.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'hackathonguide.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-10-19T00:07:48+05:00', 'author': 'fizzah', 'moddate': '2026-10-19T00:07:48+05:00', 'title': 'Microsoft Word - HackInsight_Participant_Guide_7pages', 'source': '..\\data\\pdf\\hackathonguide.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'hackathonguide.pdf', 'file_type': 'pdf'}, page_content="How to use this PDF \nBlue underlined text is a clickable link, both inside the guide and to websites. Click any entry in the Contents, or 'Go to Contents' at the top of a page, to jump. \nOpen the bookmarks panel of your PDF viewer to move between chapters. Tick the boxes and type in the blue fill-in fields. Use Adobe Acrobat, Chrome or \nEdge so that your answers are kept when you save the file. \nEach chapter also appears in the bookmarks panel. Type your answers in the blue boxes; they are saved in the file when you use Adobe Acrobat, Chrome or \nEdge. \n \nFrom a real pro

### Embeddings and VectorStoreDB

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Tesla Laptops\Downloads\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class EmbeddingManager:
  def __init__(self, model_name="all-MiniLM-L6-v2"):
    self.model_name = model_name
    self.model = None
    self._load_model()

  def _load_model(self):
    try:
      print(f"Loading embedding model : {self.model_name}...")
      self.model = SentenceTransformer(self.model_name)
      print(f"{self.model_name} model loaded successfully.")
      print(f"Dimension: {self.model.get_embedding_dimension()}")
    except Exception as e:
      print(f"Error loading model {self.model_name}: {e}")
      raise

  def generate_embeddings(self, text: List[str]) -> np.ndarray:
      if not self.model:
          raise ValueError("Model is not loaded. Please check the model initialization.")
      print(f"Generating Embeddings for {len(text)} texts...")
      embeddings =  self.model.encode(text, show_progress_bar=True)
      print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
      return embeddings

## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model : all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2556.53it/s]


all-MiniLM-L6-v2 model loaded successfully.
Dimension: 384


### VectorStore

In [12]:
class VectorStore:
  def __init__(self, collection_name="pdf_documents", persistent_directory="../data/vector_store"):
    self.collection_name = collection_name
    self.persistent_directory = persistent_directory
    self.client = None
    self.collection = None
    self._initialize_store()

  def _initialize_store(self):
    try:
      print(f"Initializing vector store at '{self.persistent_directory}'...")
      # Initialize chromadb client and collection
      os.makedirs(self.persistent_directory, exist_ok=True)
      self.client = chromadb.PersistentClient(path=self.persistent_directory)
      # Get or create the collection
      self.collection = self.client.get_or_create_collection(name=self.collection_name)
      print(f"Collection '{self.collection_name}' initialized. Existing document count: {self.collection.count()}")
    except Exception as e:
        print(f"Error initializing vector store: {e}")
        raise

  def add_documents(self, documents, embeddings):
      # Add documents and the embeddings to the vector store
      if len(documents) != len(embeddings):
          raise ValueError("Number of documents and embeddings must match.")
      print(f"Adding {len(documents)} documents to the collection '{self.collection_name}'...")

      # Prepare data for chromaDB
      ids = []
      metadatas = []
      document_texts = []
      embedding_list = []

      for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
          doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
          # Generate a unique ID for each document using UUID and the index to ensure uniqueness
          ids.append(doc_id)

          # prepare metadata
          metadata = dict(doc.metadata)  # Ensure metadata is a dictionary
          metadata['doc_index'] = i  # Add the index to the metadata for reference
          metadata['content_length'] = len(doc.page_content)  # Add content length to metadata

          metadatas.append(metadata)

          # prepare document content
          document_texts.append(doc.page_content)
          # Embedding should be converted to a list for storage in ChromaDB
          embedding_list.append(embedding.tolist())

      # Add to collection
      try:
        self.collection.add(
            ids=ids,
            embeddings=embedding_list,
            metadatas=metadatas,
            documents=document_texts
        )
        print(f"Added {len(documents)} documents to the collection.")
        print(f"Total documents in collection '{self.collection_name}': {self.collection.count()}")
      except Exception as e:
        print(f"Error adding documents to the collection: {e}")
        raise

vector_store = VectorStore()
vector_store

Initializing vector store at '../data/vector_store'...
Collection 'pdf_documents' initialized. Existing document count: 144


In [13]:
chunks

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-10-19T00:07:48+05:00', 'author': 'fizzah', 'moddate': '2026-10-19T00:07:48+05:00', 'title': 'Microsoft Word - HackInsight_Participant_Guide_7pages', 'source': '..\\data\\pdf\\hackathonguide.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'hackathonguide.pdf', 'file_type': 'pdf'}, page_content="How to use this PDF \nBlue underlined text is a clickable link, both inside the guide and to websites. Click any entry in the Contents, or 'Go to Contents' at the top of a page, to jump. \nOpen the bookmarks panel of your PDF viewer to move between chapters. Tick the boxes and type in the blue fill-in fields. Use Adobe Acrobat, Chrome or \nEdge so that your answers are kept when you save the file. \nEach chapter also appears in the bookmarks panel. Type your answers in the blue boxes; they are saved in the file when you use Adobe Acrobat, Chrome or \nEdge. \n \nFrom a real pro

In [14]:
# Convert the text into embeddings
texts = [doc.page_content for doc in chunks]

# Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in the vector database
vector_store.add_documents(chunks, embeddings)
# Stored in this hard disk

Generating Embeddings for 72 texts...


Batches: 100%|██████████| 3/3 [00:06<00:00,  2.33s/it]


Embeddings generated successfully. Shape: (72, 384)
Adding 72 documents to the collection 'pdf_documents'...
Added 72 documents to the collection.
Total documents in collection 'pdf_documents': 216


In [15]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vector_store,embedding_manager)

In [16]:
rag_retriever

In [17]:
rag_retriever.retrieve("What do we need to do in this hackathon ? ")

Retrieving documents for query: 'What do we need to do in this hackathon ? '
Top K: 5, Score threshold: 0.0
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.49it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_21b0a439_39',
  'content': 'Anything simulated or unfinished is clearly labelled. \nA short business model (user, payer, value, running cost) is ready to explain. \nYour optimization log shows what you changed, what it gained and what it cost. \nEvery team member can explain the problem, the data, the code and the results. \nDisqualification \nHard-coded outputs, faked logs, leaked test data or misrepresentation. \nYou are an architectural supervisor and technical coach for a team participating in the HackI \nYou are an architectural supervisor and technical coach for a team participating in the HackInsight 72-hour Hackathon. \n \nSTRICT CONSTRAINTS & BEHAVIORAL RULES: \n1. NO COMPLETE CODE IMPLEMENTATION: Under no circumstances should you generate fully written code, complete functions, or runnable scripts. \n2. NO DOCUMENT SUMMARIES: Do not explain, summarize, or paste text from the Participant Guide. \n3. CODE EDITOR ROLE: Act strictly as a technical logic provider dire

In [18]:
rag_retriever.retrieve("What is wellabroad project about ? ")

Retrieving documents for query: 'What is wellabroad project about ? '
Top K: 5, Score threshold: 0.0
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.39it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_40cbe68e_45',
  'content': 'WellAbroad — System Architecture &\nImplementation Guide\nAMD Developer Hackathon: ACT III · Team AFAM · Prepared 15 Oct 2026 · Audience:\nbackend/architecture owner and teammates\n1. Purpose of this document\nThis guide explains what WellAbroad does, how the system is structured, which Evolus\nand AMD pieces we use, and how each team member builds their part. It answers the\nopen questions from planning: how Evolus fits, whether we need OpenRouter or an SDK,\nand how to run the model on Fireworks AI or vLLM on AMD.\nDeadline note: the online phase on lablab.ai runs 12–17 October, followed by an on-site\nphase in Milan on 17–18 October. The working backend should be ready by Friday 16 Oct\nnight, leaving Saturday for integration, demo and submission. (Monday 19 Oct is after the\nevent.)\n2. Product summary\nWellAbroad helps tourists who face a language barrier at foreign clinics and hospitals.\nFeature 1 — Document Explainer. The user provides a

## Integration VectorDB Context Pipeline with LLM Output

In [34]:
# Simple Rag pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

# groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(api_key=groq_api_key, model="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

# Simple RAG pipeline function retrieving context and generating a response

def rag_simple(query, llm, retriever, top_k=3):
    """
    Simple RAG pipeline that retrieves context and generates a response using the LLM.
    
    Args:
        query: The input query string
        llm: The language model instance (e.g., ChatGroq)
        retriever: The RAGRetriever instance
        top_k: Number of top documents to retrieve
        score_threshold: Minimum similarity score threshold for retrieved documents"""
    # Retrieve the context documents
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([f"{doc['content']}" for doc in retrieved_docs]) if retrieved_docs else ""

    if not context:
        print("No relevant documents found for the query.")

    # Generate the answer using the LLM with the retrieved context
    prompt = f"""Use the following context to answer the question:{context}
    Question: {query}
    Answer:
    """

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [24]:
answer = rag_simple("What do we need to do in this hackathon ?", llm, rag_retriever)
print(answer)

Retrieving documents for query: 'What do we need to do in this hackathon ?'
Top K: 3, Score threshold: 0.0
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 87.78it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


**What we need to do in the HackInsight 72‑hour Hackathon**

| Step | What to deliver | Why it matters |
|------|-----------------|----------------|
| **1. Define the business model** | • **User** – who will use the product? <br>• **Payer** – who pays for it? <br>• **Value proposition** – what problem does it solve or what benefit does it provide? <br>• **Running cost** – what are the ongoing operational expenses? | A clear, concise business model lets the team and judges quickly understand the viability of the solution. |
| **2. Build a reproducible data pipeline** | • Collect and clean the required data. <br>• Document the source, format, and any preprocessing steps. | Ensures that the data can be re‑run and verified by anyone on the team or by the judges. |
| **3. Implement the core algorithm** | • Write the logic that turns data into insights or actions. <br>• Keep the code modular so each component can be tested independently. | Modular code makes it easier for every team member t

## Enhanced RAG Pipeline Features

In [31]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Gimme the WellAbroad Architecture guide briefly", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("\n\nSources:", result['sources'])
print("\nConfidence:", result['confidence'])
print("\nContext Preview:", result['context'][:300])

Retrieving documents for query: 'Gimme the WellAbroad Architecture guide briefly'
Top K: 3, Score threshold: 0.1
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.90it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **WellAbroad Architecture Guide – Quick‑Read**

| Topic | Key Points |
|-------|------------|
| **Purpose** | Explain what WellAbroad does, system structure, Evolus & AMD integration, and build steps for each team member. |
| **Product** | A language‑bridge tool for tourists in foreign medical settings. <br>• **Feature 1 – Document Explainer**: user uploads a medical document → system translates & explains it. |
| **Core Tech Stack** | • **Evolus** – data‑processing & workflow engine.<br>• **AMD** – GPU acceleration (vLLM on AMD).<br>• **Fireworks AI** – optional cloud LLM host.<br>• **OpenRouter / SDK** – routing LLM calls (choose based on latency/feature needs). |
| **Deployment Flow** | 1. **Upload** → 2. **Pre‑process** (Evolus) → 3. **LLM inference** (vLLM on AMD or Fireworks) → 4. **Post‑process & return**. |
| **Deadline** | • Online phase: 12–17 Oct (lablab.ai).<br>• Backend ready: Friday 16 Oct night.<br>• Integration/demo: Saturday 17 Oct.<br>• On‑site Milan: 17–18 Oc

In [33]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("WellAbroad Architecture guide", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("\nSummary:", result['summary'])
print("\nHistory:", result['history'][-1])

Retrieving documents for query: 'WellAbroad Architecture guide'
Top K: 3, Score threshold: 0.1
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.49it/s]

Embeddings generated successfully. Shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
WellAbroad — System Architecture &
Implementation Guide
AMD Developer Hackathon: ACT III · Team AFAM · Prepared 15 Oct 2026 · Audience:
backend/architecture owner and team

mates
1. Purpose of this document
This guide explains what WellAbroad does, how the system is structured, which Evolus
and AMD pieces we use, and how each team member builds their part. It answers the
open questions from planning: how Evolus fits, whether we need OpenRouter or an SDK,
and how to run the model on Fireworks AI or vLLM on AMD.
Deadline note: the online phase on lablab.ai runs 12–17 October, followed by an on-site
phase in Milan on 17–18 October. The working backend should be ready by Friday 16 Oct
night, leaving Saturday for integration, demo and submission. (Monday 19 Oct is after the
event.)
2. Product summary
WellAbroad helps tourists who face a language barrier at foreign clinics and hospitals.
Feature 1 — Document Explainer. The user provides a medical document by uploading a

WellAbroad — System Architecture &
Implementation Guide
AMD Developer Hackathon: ACT III · Team AFAM · Prepared 15 Oct 2026 · Audience:
backend/architecture owner and teammates
1. Purpose of th